# الدرس 01 - مقدمة في عملاء الذكاء الاصطناعي

مرحبًا بك في الدرس الأول من دورة **عملاء الذكاء الاصطناعي للمبتدئين**!

**عميل الذكاء الاصطناعي** هو برنامج يستخدم نموذجًا لغويًا كبيرًا (LLM) كمحرك تفكير يمكنه اتخاذ *إجراءات* في العالم الحقيقي — مثل استدعاء واجهات برمجة التطبيقات، أو الاستعلام من قواعد البيانات، أو تشغيل الأكواد — لتحقيق هدف نيابةً عن المستخدم.

في هذا الدفتر ستبني عميلك الأول: **وكيل سفر** يوصي بوجهات لعطلات. على طول الطريق ستتعلم كيف:

1. الاتصال بخدمة Microsoft Foundry Agent باستخدام **إطار عمل عملاء Microsoft**.
2. إعطاء الوكيل **أداة** — وظيفة Python عادية يمكنه استدعاؤها.
3. تشغيل الوكيل وفحص استجابته.
4. بث استجابة الوكيل رمزًا تلو الآخر.


## الإعداد

قبل تشغيل هذا الدفتر، تأكد من أنك:

1. **تمتلك مشروع Microsoft Foundry** مع نموذج دردشة منشور (مثلاً `gpt-5-mini`).
2. **قمت بتسجيل الدخول باستخدام Azure CLI** — نفذ الأمر `az login` في الطرفية الخاصة بك.
3. **قمت بتعيين المتغيرات البيئية المطلوبة:**
   - `AZURE_AI_PROJECT_ENDPOINT` — نقطة نهاية مشروع Microsoft Foundry الخاص بك.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — اسم النموذج الذي نشرته.

الخلية أدناه تقوم بتثبيت حزم Python التي تحتاجها.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## إنشاء وكيلك الأول

يحتاج الوكيل إلى شيئين:

- **تعليمات** تخبره *من هو* و*كيف يتصرف* (مطالبة النظام).
- **أدوات** — دوال بايثون مزينة بـ `@tool` يمكن للوكيل استدعاؤها لاسترجاع المعلومات أو أداء مهام.

فيما يلي نعرف أداة بسيطة تعيد قائمة بوجهات العطلات الشهيرة. سيستخدم الوكيل هذه الأداة عندما يطلب المستخدم توصيات السفر.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## الردود المتدفقة

من أجل تجربة أكثر تفاعلية يمكنك **بث** رد العميل. بدلاً من انتظار الرد الكامل، يقوم العميل بإعطاء أجزاء من النص أثناء توليدها. هذا مفيد بشكل خاص في واجهات الدردشة حيث تريد عرض المخرجات في الوقت الحقيقي.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## الملخص

في هذا الدرس تعلمت كيف:

- **إنشاء موفر** يتصل بخدمة Microsoft Foundry Agent عبر `FoundryChatClient`.
- **تعريف أداة** باستخدام المزخرف `@tool` بحيث يمكن للوكيل استدعاء دوال بايثون الخاصة بك.
- **تشغيل الوكيل** مع رسالة من المستخدم وطباعة استجابته.
- **بث الاستجابات** للإخراج في الوقت الحقيقي.

في الدرس التالي سوف نستكشف أطر الوكلاء بعمق أكبر ونتعلم كيفية تزويد الوكلاء بأدوات أكثر قوة وقدرات استدلال متعددة الخطوات.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنويه**:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة بالذكاء الاصطناعي [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى للدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الرسمي والمعتمد. للمعلومات الهامة، يُنصح بالاستعانة بترجمة بشرية محترفة. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
